# Bus OD Matrix Combined with OnBoard Survey Probabilities

`Input/6_9_BusProbability_ByTAZ.xlsx` (OnBoard survey) gives P(alight at `toTAZ` | board at `fromTAZ`) for the 6:00–9:00 window, with rows summing exactly to 1 per origin — including a `toTAZ = NaN` share (unknown alighting, median 7.4% where present).

1. **Probability matrix**: NaN destinations are dropped and each origin row renormalized to sum to 1 over known destinations.
2. **Combined ("new") matrix**: each origin's total from the RavKav OD matrix (`Output/bus/bus_od_taz_avg.csv` row sums, average Tuesday) is distributed over destinations by the OnBoard probabilities — RavKav sets the volumes, the OnBoard survey sets the destination pattern. Origins the OnBoard file does not cover (5.0% of volume) keep their RavKav row unchanged as a fallback.

In [1]:
import numpy as np
import pandas as pd

ob = pd.read_excel('Input/6_9_BusProbability_ByTAZ.xlsx')
assert np.allclose(ob.groupby('fromTAZ')['Probability'].sum(), 1.0)

known = ob.dropna(subset=['toTAZ']).copy()
known['toTAZ'] = known['toTAZ'].astype(int)
P = known.pivot_table(index='fromTAZ', columns='toTAZ', values='Probability', aggfunc='sum').fillna(0)
row_sums = P.sum(axis=1)
P = P[row_sums > 0].div(row_sums[row_sums > 0], axis=0)   # renormalize over known destinations
P.index = P.index.astype(int)
P.index.name = 'fromTAZ'
P.to_csv('Output/bus/bus_probability_matrix.csv', float_format='%.6g')
dropped = (row_sums <= 0).sum()
print(f"probability matrix: {P.shape}, rows sum to 1 (renormalized; unknown-destination share removed)")
print(f"origins dropped for having only unknown destinations: {dropped}")

probability matrix: (594, 548), rows sum to 1 (renormalized; unknown-destination share removed)
origins dropped for having only unknown destinations: 0


In [2]:
rav = pd.read_csv('Output/bus/bus_od_taz_avg.csv', index_col=0)
rav.index = rav.index.astype(int)
rav.columns = rav.columns.astype(int)

all_dest = sorted(set(rav.columns) | set(P.columns))
new = pd.DataFrame(0.0, index=rav.index, columns=all_dest)
vol = rav.sum(axis=1)

onboard_vol = fallback_vol = 0.0
for o in rav.index:
    if o in P.index:
        new.loc[o, P.columns] = vol[o] * P.loc[o].values
        onboard_vol += vol[o]
    else:
        new.loc[o, rav.columns] = rav.loc[o].values
        fallback_vol += vol[o]

new.index.name = 'orig_taz'
new.to_csv('Output/bus/bus_od_taz_new.csv', float_format='%.6g')
assert abs(new.values.sum() - rav.values.sum()) < 0.5
print(f"new matrix: {new.shape}, total {new.values.sum():,.0f} passengers/avg Tuesday (preserved)")
print(f"volume redistributed by OnBoard probabilities: {onboard_vol:,.0f} ({onboard_vol / vol.sum():.1%}) | "
      f"RavKav fallback: {fallback_vol:,.0f} ({fallback_vol / vol.sum():.1%})")

# how different are the two destination patterns?
common_o = [o for o in rav.index if o in P.index]
common_d = sorted(set(rav.columns) & set(P.columns))
a = rav.loc[common_o, common_d].div(rav.loc[common_o].sum(axis=1), axis=0).fillna(0)
b = new.loc[common_o, common_d].div(new.loc[common_o].sum(axis=1), axis=0).fillna(0)
r = np.corrcoef(a.values.flatten(), b.values.flatten())[0, 1]
print(f"correlation between RavKav and OnBoard destination distributions (common cells): r = {r:.4f}")

new matrix: (722, 728), total 143,402 passengers/avg Tuesday (preserved)
volume redistributed by OnBoard probabilities: 135,533 (94.5%) | RavKav fallback: 7,869 (5.5%)
correlation between RavKav and OnBoard destination distributions (common cells): r = 0.0846


## Notes

- The "new" matrix keeps RavKav's origin volumes (total 143,402 average-Tuesday passengers, asserted preserved) and replaces the destination pattern with the OnBoard survey's probabilities wherever available.
- The OnBoard file's unknown-alighting share (`toTAZ` = NaN) is removed by row renormalization — i.e., unknown alightings are assumed to distribute like the known ones of the same origin.
- Origins without OnBoard coverage (142 zones, 5.0% of volume) and origins whose OnBoard row is entirely unknown-destination keep their RavKav distribution.